<a href="https://colab.research.google.com/github/Ashwin-2408/Tensor_Flow_Learning/blob/main/notebooks/Transfer_Learning_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning

Transfer Learning is leveraging a  working existing model architecture and learned patterns for our problem

## Downloading Data

In [1]:
! wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

--2025-06-19 17:15:51--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.68.207, 142.250.4.207, 74.125.24.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.68.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip’

10_food_classes_10_ 100%[===================>] 160.74M  23.3MB/s    in 8.2s    

2025-06-19 17:16:01 (19.6 MB/s) - ‘10_food_classes_10_percent.zip’ saved [168546183/168546183]



In [2]:
import zipfile
zipref=zipfile.ZipFile("/content/10_food_classes_10_percent.zip")
zipref.extractall()
zipref.close()

In [3]:
import os

for dirpath,dirnames,filenames in os.walk("/content/10_food_classes_10_percent"):
  print(f"There are {len(dirnames)} directories and {len(filenames)} images in the directory '{dirpath}'.")

There are 2 directories and 0 images in the directory '/content/10_food_classes_10_percent'.
There are 10 directories and 0 images in the directory '/content/10_food_classes_10_percent/test'.
There are 0 directories and 250 images in the directory '/content/10_food_classes_10_percent/test/ramen'.
There are 0 directories and 250 images in the directory '/content/10_food_classes_10_percent/test/ice_cream'.
There are 0 directories and 250 images in the directory '/content/10_food_classes_10_percent/test/hamburger'.
There are 0 directories and 250 images in the directory '/content/10_food_classes_10_percent/test/steak'.
There are 0 directories and 250 images in the directory '/content/10_food_classes_10_percent/test/grilled_salmon'.
There are 0 directories and 250 images in the directory '/content/10_food_classes_10_percent/test/fried_rice'.
There are 0 directories and 250 images in the directory '/content/10_food_classes_10_percent/test/chicken_wings'.
There are 0 directories and 250 imag

## Preparing the Data

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE=32
IMAGE_SIZE=[224,224]

Train_Data_Generator=ImageDataGenerator(rescale=1/255.)
Test_Data_Generator=ImageDataGenerator(rescale=1/255.)

Train_Directory="/content/10_food_classes_10_percent/train"
Test_Directory="/content/10_food_classes_10_percent/test"

Train_Data=Train_Data_Generator.flow_from_directory(Train_Directory,batch_size=BATCH_SIZE,target_size=IMAGE_SIZE,class_mode="categorical")
Test_Data=Test_Data_Generator.flow_from_directory(Test_Directory,batch_size=BATCH_SIZE,target_size=IMAGE_SIZE,class_mode="categorical")

Found 750 images belonging to 10 classes.
Found 2500 images belonging to 10 classes.


## Setting up Callbacks : Things to run when our model trains

Callbacks are extra functionality that we can add to our models,whilst they train or after training.Popular Callbacks are:

* Tracking Experiments with the TensorBoard CallBack
* Model checkpoint with the ModelCheckPoint Callback
* Stopping a model from training(before it runs long and overfits) with the EarlyStopping Callback

In [16]:
#Create TensorBoardCallback (functionized because we need to create a new_one for each model)
import datetime

def create_tensor_board_callback(dir_name,experiment_name):
  log_dir=dir_name +"/" + experiment_name +"/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
  tensorboard_callback=tf_keras.callbacks.TensorBoard(log_dir=log_dir)
  print(f"Saving tensorboard log files to : {log_dir}")
  return tensorboard_callback

## Creating Models Using Tensorflow Hub




In [6]:
#comparing the two models

res_net_model_url = "https://www.kaggle.com/models/google/resnet-v2/TensorFlow2/50-classification/2"
efficient_net_url="https://www.kaggle.com/models/tensorflow/efficientnet/TensorFlow2/b0-classification/1"

In [7]:
import tensorflow as tf
import tensorflow_hub as hub
import tf_keras


In [8]:
# Let's create a create_model() function to create a model from a URL
import tensorflow as tf
import tensorflow_hub as hub

# Make sure you're using the right imports
def create_model(model_url, max_classes=10):
    feature_extractor_layer = hub.KerasLayer(
        model_url,
        trainable=False,
        name="feature_extraction_layer",
        input_shape=(224, 224, 3)
    )

    model = tf_keras.Sequential([
        feature_extractor_layer,
        tf_keras.layers.Flatten(),
        tf_keras.layers.Dense(max_classes, activation="softmax", name="output_layer")
    ])
    return model


In [10]:
res_net_model=create_model(res_net_model_url)

In [17]:
res_net_model.compile(loss=tf.keras.losses.CategoricalCrossentropy(),optimizer=tf_keras.optimizers.Adam(),metrics=["accuracy"])
res_net_model.fit(Train_Data,epochs=5,steps_per_epoch=len(Train_Data),validation_data=Test_Data,callbacks=[create_tensor_board_callback(dir_name="tensorhub",experiment_name="resnet")])

Saving tensorboard log files to : tensorhub/resnet/20250619-172057
Epoch 1/5
24/24 [==============================] - 16s 618ms/step - loss: 0.3750 - accuracy: 0.8800 - val_loss: 0.7538 - val_accuracy: 0.7500
Epoch 2/5
24/24 [==============================] - 12s 537ms/step - loss: 0.3248 - accuracy: 0.9013 - val_loss: 0.7974 - val_accuracy: 0.7484
Epoch 3/5
24/24 [==============================] - 10s 437ms/step - loss: 0.2441 - accuracy: 0.9387 - val_loss: 0.7317 - val_accuracy: 0.7696
Epoch 4/5
24/24 [==============================] - 10s 432ms/step - loss: 0.1986 - accuracy: 0.9640 - val_loss: 0.7966 - val_accuracy: 0.7560
Epoch 5/5
24/24 [==============================] - 10s 446ms/step - loss: 0.1688 - accuracy: 0.9680 - val_loss: 0.7535 - val_accuracy: 0.7628


In [18]:
res_net_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 feature_extraction_layer (  (None, 1001)              25615849  
 KerasLayer)                                                     
                                                                 
 flatten (Flatten)           (None, 1001)              0         
                                                                 
 output_layer (Dense)        (None, 10)                10020     
                                                                 
Total params: 25625869 (97.75 MB)
Trainable params: 10020 (39.14 KB)
Non-trainable params: 25615849 (97.72 MB)
_________________________________________________________________
